In [1]:
# ============================================================
# NOTEBOOK 00: INGESTA Y CONSOLIDACIÓN
# Proyecto: SIS-DEMAND Forecast
# Descripción: Carga los 9 semestres de datos SIS (.csv, UTF-8),
#              estandariza columnas, valida integridad y
#              genera un único dataset consolidado en parquet.
# Input:  data/raw/2021_S1.csv ... data/raw/2025_S1.csv
# Output: data/processed/sis_consolidado.parquet
# Última actualización: 2026-04-25
# ============================================================

import warnings; warnings.filterwarnings('ignore')
import logging; logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from unidecode import unidecode

## 1. Configuración — Mapeo de semestres a PERIODO_NUM

In [3]:
# Mapeo de etiqueta de semestre a número de periodo (1=2021S1, 2=2021S2, ...)
SEMESTRES = {
    '2021_S1': 1, '2021_S2': 2,
    '2022_S1': 3, '2022_S2': 4,
    '2023_S1': 5, '2023_S2': 6,
    '2024_S1': 7, '2024_S2': 8,
    '2025_S1': 9
}

RAW_DIR = Path('../data/raw')
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

logging.info(f"Directorio de datos crudos: {RAW_DIR.resolve()}")
archivos_disponibles = sorted(RAW_DIR.glob('*.csv'))
logging.info(f"Archivos .csv encontrados: {len(archivos_disponibles)}")
for f in archivos_disponibles:
    size_mb = f.stat().st_size / 1024**2
    logging.info(f"  {f.name}  ({size_mb:.0f} MB)")

INFO — Directorio de datos crudos: C:\PROYECTO ATENCIONES CENTRUM PUCP\files\data\raw
INFO — Archivos .csv encontrados: 9
INFO —   2021_S1.csv  (629 MB)
INFO —   2021_S2.csv  (729 MB)
INFO —   2022_S1.csv  (811 MB)
INFO —   2022_S2.csv  (845 MB)
INFO —   2023_S1.csv  (963 MB)
INFO —   2023_S2.csv  (960 MB)
INFO —   2024_S1.csv  (1029 MB)
INFO —   2024_S2.csv  (1003 MB)
INFO —   2025_S1.csv  (1076 MB)


## 2. Función de carga y estandarización

In [4]:
def leer_y_estandarizar(path: str, semestre_label: str) -> pd.DataFrame:
    """Carga un semestre SIS (CSV UTF-8) y lo estandariza: columnas, tipos y texto."""
    path = Path(path)
    size_mb = path.stat().st_size / 1024**2
    logging.info(f"Leyendo {path.name} ({size_mb:.0f} MB) ...")

    # Leer como strings para preservar ceros en códigos; low_memory=False evita warnings en archivos grandes
    for enc in ['utf-8', 'utf-8-sig', 'latin-1']:
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc, low_memory=False)
            logging.info(f"  Encoding OK: {enc}")
            break
        except UnicodeDecodeError:
            continue

    # Normalizar nombres de columnas: sin tildes, mayúsculas, sin espacios
    df.columns = [unidecode(c).upper().strip().replace(' ', '_') for c in df.columns]

    # ANO puede llegar como 'AO' tras normalización de 'AÑO'
    if 'ANO' not in df.columns and 'AO' in df.columns:
        df.rename(columns={'AO': 'ANO'}, inplace=True)

    # Casteos obligatorios con manejo de errores
    df['ATENCIONES'] = pd.to_numeric(df['ATENCIONES'], errors='coerce').fillna(0).astype('int32')
    df['ANO']        = pd.to_numeric(df.get('ANO', 0), errors='coerce').fillna(0).astype('int16')
    df['MES']        = pd.to_numeric(df['MES'], errors='coerce').fillna(0).astype('int8')

    # UBIGEO: rellenar con ceros a la izquierda hasta 6 dígitos
    df['UBIGEO_DISTRITO'] = df['UBIGEO_DISTRITO'].str.strip().str.zfill(6)

    # Limpiar encoding residual (tildes, eñes) en columnas de texto libre
    # PLAN_SEGURO es el nombre real en los CSV (no PLAN_DE_SEGURO)
    cols_texto = ['REGION', 'PROVINCIA', 'DISTRITO', 'DESC_SERVICIO',
                  'DESC_UNIDAD_EJECUTORA', 'IPRESS', 'NIVEL_EESS',
                  'GRUPO_EDAD', 'SEXO', 'PLAN_SEGURO']
    for col in cols_texto:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: unidecode(str(x)).upper().strip())

    # Optimización de memoria: categorías antes de guardar en la lista
    cols_cat = ['REGION', 'PROVINCIA', 'NIVEL_EESS', 'DESC_SERVICIO',
                'GRUPO_EDAD', 'SEXO', 'PLAN_SEGURO', 'DESC_UNIDAD_EJECUTORA']
    for col in cols_cat:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Columnas de identificación de periodo
    df['SEMESTRE_LABEL'] = semestre_label
    df['PERIODO_NUM']    = np.int8(SEMESTRES[semestre_label])
    df['SEMESTRE']       = np.int8(1 if 'S1' in semestre_label else 2)

    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    logging.info(f"  {path.name}: {df.shape[0]:,} filas, {df.shape[1]} cols — {mem_mb:.0f} MB en RAM")
    return df

## 3. Consolidación de todos los semestres disponibles

In [5]:
dfs = []
semestres_cargados = []
semestres_faltantes = []

for label, num in SEMESTRES.items():
    path = RAW_DIR / f'{label}.csv'
    if path.exists():
        df_sem = leer_y_estandarizar(str(path), label)
        dfs.append(df_sem)
        semestres_cargados.append(label)
    else:
        semestres_faltantes.append(label)
        logging.warning(f"Archivo no encontrado: {path.name}")

if not dfs:
    raise FileNotFoundError(
        f"No se encontró ningún archivo .csv en {RAW_DIR.resolve()}\n"
        "Guarda los archivos en data/raw/ con los nombres: 2021_S1.csv, 2021_S2.csv, ..."
    )

logging.info(f"Semestres cargados ({len(semestres_cargados)}): {semestres_cargados}")
if semestres_faltantes:
    logging.warning(f"Semestres FALTANTES: {semestres_faltantes}")

INFO — Leyendo 2021_S1.csv (629 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2021_S1.csv: 3,338,348 filas, 20 cols — 1505 MB en RAM
INFO — Leyendo 2021_S2.csv (729 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2021_S2.csv: 3,832,485 filas, 20 cols — 1732 MB en RAM
INFO — Leyendo 2022_S1.csv (811 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2022_S1.csv: 4,247,280 filas, 20 cols — 1921 MB en RAM
INFO — Leyendo 2022_S2.csv (845 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2022_S2.csv: 4,418,825 filas, 20 cols — 2000 MB en RAM
INFO — Leyendo 2023_S1.csv (963 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2023_S1.csv: 5,024,801 filas, 20 cols — 2274 MB en RAM
INFO — Leyendo 2023_S2.csv (960 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2023_S2.csv: 5,001,987 filas, 20 cols — 2264 MB en RAM
INFO — Leyendo 2024_S1.csv (1029 MB) ...
INFO —   Encoding OK: utf-8
INFO —   2024_S1.csv: 5,352,228 filas, 20 cols — 2424 MB en RAM
INFO — Leyendo 2024_S2.csv (1003 MB) ...
INFO —   Encoding OK: utf-8
INFO —

In [6]:
# Concatenar todos los semestres en un único DataFrame
df_raw = pd.concat(dfs, ignore_index=True)
del dfs  # liberar memoria intermedia
logging.info(f"Dataset consolidado: {df_raw.shape[0]:,} filas, {df_raw.shape[1]} columnas")

INFO — Dataset consolidado: 41,997,568 filas, 20 columnas


## 4. Diagnóstico de columnas y esquema

In [7]:
# Revisar columnas disponibles y su completitud
print("=" * 60)
print("COLUMNAS DEL DATASET CONSOLIDADO")
print("=" * 60)
resumen = pd.DataFrame({
    'dtype':     df_raw.dtypes,
    'nulos':     df_raw.isnull().sum(),
    'pct_nulos': (df_raw.isnull().mean() * 100).round(2),
    'n_unicos':  df_raw.nunique()
})
print(resumen.to_string())

print(f"\nPeriodos encontrados: {sorted(df_raw['PERIODO_NUM'].unique())}")
print(f"\nDistribución por semestre:")
print(df_raw.groupby(['SEMESTRE_LABEL', 'PERIODO_NUM']).size().reset_index(name='n_filas').to_string(index=False))

COLUMNAS DEL DATASET CONSOLIDADO
                          dtype  nulos  pct_nulos  n_unicos
ANO                       int16      0        0.0         5
MES                        int8      0        0.0        12
REGION                 category      0        0.0        26
PROVINCIA                object      0        0.0       197
UBIGEO_DISTRITO          object      0        0.0      1889
DISTRITO                 object      0        0.0      1735
COD_UNIDAD_EJECUTORA     object    606        0.0       225
DESC_UNIDAD_EJECUTORA    object      0        0.0       227
COD_IPRESS               object      0        0.0      8725
IPRESS                   object      0        0.0      8952
NIVEL_EESS             category      0        0.0         4
PLAN_SEGURO            category      0        0.0         5
COD_SERVICIO             object      0        0.0        64
DESC_SERVICIO            object      0        0.0        65
SEXO                   category      0        0.0         2
GRUPO_E

## 5. Validaciones de integridad

In [8]:
errores = []

# V1: No atenciones negativas
neg = (df_raw['ATENCIONES'] < 0).sum()
if neg > 0:
    errores.append(f"V1 FAIL: {neg:,} registros con ATENCIONES negativas")
else:
    logging.info("V1 OK: No hay atenciones negativas")

# V2: UBIGEO siempre con 6 dígitos
mal_ubigeo = df_raw['UBIGEO_DISTRITO'].str.len().ne(6).sum()
if mal_ubigeo > 0:
    errores.append(f"V2 FAIL: {mal_ubigeo:,} UBIGEOs sin 6 dígitos")
else:
    logging.info("V2 OK: Todos los UBIGEOs tienen 6 dígitos")

# V3: Columnas clave sin nulos
for col in ['ATENCIONES', 'UBIGEO_DISTRITO', 'DESC_SERVICIO']:
    if col in df_raw.columns:
        n_nulos = df_raw[col].isnull().sum()
        if n_nulos > 0:
            errores.append(f"V3 FAIL: {n_nulos:,} nulos en columna clave '{col}'")
        else:
            logging.info(f"V3 OK: '{col}' sin nulos")

# V4: 9 periodos esperados
periodos = df_raw['PERIODO_NUM'].nunique()
logging.info(f"V4: {periodos} periodo(s) cargados de 9 esperados")

# Reporte final
if errores:
    print("\n[ADVERTENCIAS DE VALIDACIÓN]")
    for e in errores:
        print(f"  {e}")
else:
    print("\n✓ Todas las validaciones pasaron correctamente")

INFO — V1 OK: No hay atenciones negativas
INFO — V2 OK: Todos los UBIGEOs tienen 6 dígitos
INFO — V3 OK: 'ATENCIONES' sin nulos
INFO — V3 OK: 'UBIGEO_DISTRITO' sin nulos
INFO — V3 OK: 'DESC_SERVICIO' sin nulos
INFO — V4: 9 periodo(s) cargados de 9 esperados



✓ Todas las validaciones pasaron correctamente


## 6. Optimización de memoria final

In [9]:
mem_antes = df_raw.memory_usage(deep=True).sum() / 1024**2

# Re-aplicar category al dataset concatenado (el concat puede desoptimizar tipos)
cols_category = ['REGION', 'PROVINCIA', 'NIVEL_EESS', 'DESC_SERVICIO',
                 'GRUPO_EDAD', 'SEXO', 'PLAN_SEGURO', 'SEMESTRE_LABEL',
                 'DESC_UNIDAD_EJECUTORA']
for col in cols_category:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype('category')

mem_despues = df_raw.memory_usage(deep=True).sum() / 1024**2
logging.info(f"Memoria: {mem_antes:.0f} MB → {mem_despues:.0f} MB (ahorro: {(1 - mem_despues/mem_antes)*100:.1f}%)")

INFO — Memoria: 28900 MB → 16482 MB (ahorro: 43.0%)


## 7. Estadísticas descriptivas básicas

In [10]:
print("=" * 60)
print("ESTADÍSTICAS DE ATENCIONES")
print("=" * 60)
print(df_raw['ATENCIONES'].describe(percentiles=[.25, .50, .75, .90, .95, .99]).round(2).to_string())

print(f"\nTotal atenciones consolidadas: {df_raw['ATENCIONES'].sum():,}")
print(f"Registros con ATENCIONES = 0: {(df_raw['ATENCIONES'] == 0).sum():,} "
      f"({(df_raw['ATENCIONES'] == 0).mean()*100:.1f}%)")

if 'REGION' in df_raw.columns:
    print(f"\nTop 5 regiones por atenciones:")
    print(df_raw.groupby('REGION')['ATENCIONES'].sum()
          .sort_values(ascending=False).head().to_string())

if 'DESC_SERVICIO' in df_raw.columns:
    print(f"\nTop 5 servicios por atenciones:")
    print(df_raw.groupby('DESC_SERVICIO')['ATENCIONES'].sum()
          .sort_values(ascending=False).head().to_string())

ESTADÍSTICAS DE ATENCIONES
count    41997568.00
mean            8.42
std            31.72
min             1.00
25%             1.00
50%             3.00
75%             7.00
90%            17.00
95%            30.00
99%            85.00
max          9923.00

Total atenciones consolidadas: 353,696,545
Registros con ATENCIONES = 0: 0 (0.0%)

Top 5 regiones por atenciones:
REGION
LIMA METROPOLITANA    63262786
CAJAMARCA             27246308
ANCASH                19154654
CUSCO                 18651968
LA LIBERTAD           18188465

Top 5 servicios por atenciones:
DESC_SERVICIO
CONSULTA EXTERNA                                                104561728
DETECCION DE PROBLEMAS EN SALUD MENTAL                           26530782
CONTROL DE CRECIMIENTO Y DESARROLLO EN MENORES DE 0 - 4 ANOS     26407224
APOYO AL DIAGNOSTICO                                             22183662
SALUD REPRODUCTIVA (PLANIFICACION FAMILIAR)                      18220784


## 8. Guardar dataset consolidado en Parquet

In [ ]:
output_path = OUT_DIR / 'sis_consolidado.parquet'
df_raw.to_parquet(output_path, index=False, compression='snappy')
logging.info(f"Guardado: {output_path}")
logging.info(f"Guardado: {df_raw.shape[0]:,} filas, {df_raw.shape[1]} columnas")

# Verificar que el archivo se puede leer correctamente
df_check = pd.read_parquet(output_path)
assert df_check.shape == df_raw.shape, "Error: el archivo parquet no coincide"
logging.info(f"Verificación OK: {df_check.shape[0]:,} filas leídas desde parquet")

size_mb = output_path.stat().st_size / 1024**2
logging.info(f"Tamaño del archivo: {size_mb:.0f} MB")

print(f"\n{'='*60}")
print("NOTEBOOK 00 COMPLETADO")
print(f"  Output:   {output_path}")
print(f"  Filas:    {df_raw.shape[0]:,}")
print(f"  Columnas: {df_raw.shape[1]}")
print(f"  Periodos: {df_raw['PERIODO_NUM'].min()} → {df_raw['PERIODO_NUM'].max()}")
print(f"  Tamaño:   {size_mb:.0f} MB")
print(f"{'='*60}")
print("Siguiente paso: notebooks/01_eda_univariado.ipynb")

INFO — Guardado: ..\data\processed\sis_consolidado.parquet
INFO — Guardado: 41,997,568 filas, 20 columnas
INFO — Verificación OK: 41,997,568 filas leídas desde parquet
INFO — Tamaño del archivo: 520 MB



NOTEBOOK 00 COMPLETADO
  Output:   ..\data\processed\sis_consolidado.parquet
  Filas:    41,997,568
  Columnas: 20
  Periodos: 1 → 9
  Tamaño:   520 MB
Siguiente paso: notebooks/01_eda_univariado.ipynb


: 